# G2 · D6-TV1 Adapter Preflight — ViT-B/16

**Contract v3 / PROMPT 03:** kiểm thử kỹ thuật 9 cặp `r=[64,128,256]`, `d=[256,512,768]`.
`r` là bottleneck, `d` là projection **bên trong** Adapter; đầu ra giữ width **768**. `fusion.dim=64` độc lập.

CPU dùng feature mock nhỏ `[B,768,4,4]` và E2/Adapter/Decoder/loss/train-step thật; kiểm tra shape, hai bước backward, gradient sau gamma=0 và số tham số.
GPU tùy chọn dùng **DINOv3 ViT-B/16 pretrained thật**, Local 512×512 → feature 32×32, để smoke và đo VRAM/OOM.

Không dùng dataset/DEV/TEST, không AU-PRO hoặc xếp hạng, không 72 jobs, không main training và không tạo selection lock. Kết quả chỉ là tương thích kỹ thuật; lựa chọn r/d/Fusion vẫn **pending joint selection D10**.

Chạy các cell CPU từ trên xuống. Để smoke GPU, bật `RUN_GPU_SMOKE` và chạy cell GPU riêng. Kết quả chưa đo phải ghi **NOT RUN**.


## 1. Setup: Git, dependencies và Google Drive

In [ ]:
# @title Input và ngân sách GPU
from pathlib import Path
import os, sys, json, hashlib, shutil, subprocess

GITHUB_URL = 'https://github.com/haduckien-raccoon/msila.git'
GITHUB_REF = 'g2/member1'
GIT_COMMIT = ''  # Đặt SHA đã chứa notebook/helper mới để tái lập chính xác.
DINO_URL = 'https://github.com/facebookresearch/dinov3.git'
DINO_COMMIT = 'main'  # Resolve thành SHA và ghi lại; đặt SHA cụ thể để resume.
CHECKPOINT_ON_DRIVE = Path('/content/drive/MyDrive/msila/models/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth')

RUN_GPU_SMOKE = False  # Chỉ bật khi muốn chi ngân sách GPU.
GPU_PAIRS = ['256:768']  # Cặp lớn nhất mặc định; ['all'] kiểm thử GPU cả 9 cặp.
MAX_BATCH = 128  # Trần probe: T4≤16, L4≤32, A100 40GB≤64 / 80GB≤128.
MEMORY_FRACTION = 0.80  # Chừa VRAM; đo thực tế và giảm batch nếu OOM/vượt dự phòng.
RESUME_GPU = True  # Skip chỉ khi evidence/config/checkpoint/hardware tương thích.

In [ ]:
# @title OUTPUT — chỉ thay đường dẫn và RUN_ID tại cell này
DRIVE_OUTPUT_BASE = Path('/content/drive/MyDrive/msila/outputs/G2/D6')
RUN_ID = 'adapter_preflight_v3_vitb16'
DRIVE_OUTPUT_ROOT = DRIVE_OUTPUT_BASE / RUN_ID
LOCAL_WORK = Path('/content/msila_d6_preflight') / RUN_ID
LOCAL_OUTPUT_ROOT = LOCAL_WORK / 'outputs'
REPO_DIR = LOCAL_WORK / 'repo'
DINO_DIR = LOCAL_WORK / 'dinov3'
LOCAL_CHECKPOINT = LOCAL_WORK / 'weights' / CHECKPOINT_ON_DRIVE.name
VENV_DIR = LOCAL_WORK / 'venv'
MODEL_CONFIG = LOCAL_WORK / 'g2_preflight.yaml'

In [ ]:
# @title Mount Drive, checkout Git và cài dependencies
from google.colab import drive
drive.mount('/content/drive')
for directory in (LOCAL_WORK, LOCAL_OUTPUT_ROOT, DRIVE_OUTPUT_ROOT, LOCAL_WORK / 'tmp', LOCAL_WORK / 'mpl'):
    directory.mkdir(parents=True, exist_ok=True)

def checkout_pinned(url, directory, ref):
    if not (directory / '.git').exists():
        subprocess.run(['git', 'clone', '--depth', '1', url, str(directory)], check=True)
    dirty = subprocess.check_output(['git', '-C', str(directory), 'status', '--porcelain'], text=True)
    if dirty.strip():
        raise RuntimeError(f'Repo có thay đổi chưa lưu: {directory}. Lưu thay đổi trước khi checkout.')
    subprocess.run(['git', '-C', str(directory), 'fetch', '--depth', '1', 'origin', ref], check=True)
    commit = subprocess.check_output(['git', '-C', str(directory), 'rev-parse', 'FETCH_HEAD'], text=True).strip()
    subprocess.run(['git', '-C', str(directory), 'checkout', '--detach', commit], check=True)
    return commit

RESOLVED_COMMIT = checkout_pinned(GITHUB_URL, REPO_DIR, GIT_COMMIT or GITHUB_REF)
if not (REPO_DIR / 'scripts/g2_adapter_preflight.py').is_file():
    raise RuntimeError('Commit chưa có D6 preflight mới. Commit/push thay đổi lên g2/member1, rồi đặt GIT_COMMIT và chạy lại.')
print('MS-ILA commit:', RESOLVED_COMMIT)

# Kế thừa CUDA torch của Colab; dependencies chạy trong process riêng.
if not (VENV_DIR / 'bin/python').is_file():
    subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', '--without-pip', str(VENV_DIR)], check=True)
PYTHON = str(VENV_DIR / 'bin/python')
requirements = [line for line in (REPO_DIR / 'requirements.txt').read_text().splitlines()
                if line.strip() and not line.lstrip().startswith(('#', 'torch', 'safetensors'))]
requirements.append('safetensors>=0.8')
dependency_file = LOCAL_WORK / 'requirements_colab.txt'
dependency_file.write_text('\n'.join(requirements) + '\n')
subprocess.run([PYTHON, '-m', 'pip', 'install', '-q', '-r', str(dependency_file)], check=True)
RUN_ENV = dict(os.environ, PYTHONUNBUFFERED='1', PYTEST_DISABLE_PLUGIN_AUTOLOAD='1',
               CUBLAS_WORKSPACE_CONFIG=':4096:8', MPLCONFIGDIR=str(LOCAL_WORK / 'mpl'), TMPDIR=str(LOCAL_WORK / 'tmp'))
subprocess.run([PYTHON, '-c', "import torch,safetensors; from packaging.version import Version; assert Version(safetensors.__version__) >= Version('0.8'); print('torch',torch.__version__,'safetensors',safetensors.__version__,'GPU available:',torch.cuda.is_available())"],
               env=RUN_ENV, check=True)

## 2. CPU preflight bắt buộc: 9 cặp + invalid config + hash

In [ ]:
# @title Resolved config và khôi phục evidence để resume
import yaml
cfg = yaml.safe_load((REPO_DIR / 'configs/g2_experiments.yaml').read_text())
cfg['backbone'].update(name='dinov3_vitb16', repo_dir=str(DINO_DIR), weights=str(LOCAL_CHECKPOINT))
cfg['decoder']['deterministic_resize'] = True
MODEL_CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False))

previous = DRIVE_OUTPUT_ROOT / 'setup_manifest.json'
if previous.is_file() and json.loads(previous.read_text())['commit'] != RESOLVED_COMMIT:
    raise RuntimeError('RUN_ID chứa commit khác. Chọn RUN_ID mới hoặc đặt GIT_COMMIT theo manifest cũ.')
if DRIVE_OUTPUT_ROOT.exists():
    shutil.copytree(DRIVE_OUTPUT_ROOT, LOCAL_OUTPUT_ROOT, dirs_exist_ok=True)
setup = dict(json.loads(previous.read_text()) if previous.is_file() else {},
             commit=RESOLVED_COMMIT, branch=GITHUB_REF, run_id=RUN_ID,
             scope='D6 technical preflight', gpu_requested=RUN_GPU_SMOKE,
             selection_status='pending_joint_selection_D10')
(LOCAL_OUTPUT_ROOT / 'setup_manifest.json').write_text(json.dumps(setup, indent=2))
shutil.copy2(LOCAL_OUTPUT_ROOT / 'setup_manifest.json', DRIVE_OUTPUT_ROOT / 'setup_manifest.json')

HELPER = str(REPO_DIR / 'scripts/g2_adapter_preflight.py')
def run_command(arguments):
    # Stream logs and persist even when a subprocess fails.
    with (LOCAL_OUTPUT_ROOT / 'colab.log').open('a') as log:
        process = subprocess.Popen(arguments, cwd=REPO_DIR, env=RUN_ENV, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end=''); log.write(line); log.flush()
        result = process.wait()
    shutil.copy2(LOCAL_OUTPUT_ROOT / 'colab.log', DRIVE_OUTPUT_ROOT / 'colab.log')
    if result:
        raise RuntimeError(f'Command failed ({result}); xem logs/evidence trên Drive.')

COMMON = ['--model-config', str(MODEL_CONFIG), '--output-root', str(LOCAL_OUTPUT_ROOT),
          '--drive-root', str(DRIVE_OUTPUT_ROOT)]
run_command([PYTHON, '-m', 'pytest', '-q', 'tests/test_g2_adapter_preflight.py',
             'tests/test_adapter_factory.py', 'tests/test_g2_tv1_model.py', '-k', 'not real_e2'])
run_command([PYTHON, HELPER, '--action', 'cpu', *COMMON])

## 3. GPU smoke tùy chọn — backbone thật, peak VRAM/OOM

Mỗi cặp chỉ chạy **hai train-step kiểm tra** trên input fixture 512×512. Không chạy epoch/dataset và không gọi `train_e2` full loop.
Backbone chỉ load một lần trong process GPU; mỗi cặp/batch probe có Adapter, Decoder và optimizer mới.
Batch khởi đầu theo VRAM thực, giảm theo lũy thừa 2 đến 1 nếu OOM hoặc vượt dự phòng. Các trial được lưu, nên OOM được báo rõ, không bị đổi thành PASS kỹ thuật giả.

Để đo tất cả 9 cặp: đặt `GPU_PAIRS=['all']`. Để tiết kiệm: giữ một cặp hoặc danh sách `['64:256','256:768']`.
Đổi GPU/commit/checkpoint/cấu hình thì dùng RUN_ID mới. Cặp đã PASS GPU chỉ skip nếu resume hợp lệ.


In [ ]:
# @title Chạy riêng GPU smoke (mặc định tắt)
if RUN_GPU_SMOKE:
    gpu_check = "import torch; assert torch.cuda.is_available(), 'GPU NOT RUN: chọn T4/L4/A100'; print(torch.cuda.get_device_name(0))"
    subprocess.run([PYTHON, '-c', gpu_check], env=RUN_ENV, check=True)
    RESOLVED_DINO_COMMIT = checkout_pinned(DINO_URL, DINO_DIR, DINO_COMMIT)
    if not CHECKPOINT_ON_DRIVE.is_file():
        raise FileNotFoundError(f'Chưa có checkpoint ViT-B/16 trên Drive: {CHECKPOINT_ON_DRIVE}')
    def sha256_file(path):
        digest = hashlib.sha256()
        with path.open('rb') as file:
            for chunk in iter(lambda: file.read(8 * 1024 * 1024), b''):
                digest.update(chunk)
        return digest.hexdigest()
    checkpoint_sha = sha256_file(CHECKPOINT_ON_DRIVE)
    setup.update(dino_commit=RESOLVED_DINO_COMMIT, checkpoint_sha256=checkpoint_sha)
    old_setup = json.loads((DRIVE_OUTPUT_ROOT / 'setup_manifest.json').read_text())
    for key in ('dino_commit', 'checkpoint_sha256'):
        if old_setup.get(key) and old_setup[key] != setup[key]:
            raise RuntimeError(f'{key} thay đổi; dùng RUN_ID mới để giữ lịch sử.')
    LOCAL_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    if not LOCAL_CHECKPOINT.is_file() or sha256_file(LOCAL_CHECKPOINT) != checkpoint_sha:
        shutil.copy2(CHECKPOINT_ON_DRIVE, LOCAL_CHECKPOINT)
    assert sha256_file(LOCAL_CHECKPOINT) == checkpoint_sha
    (LOCAL_OUTPUT_ROOT / 'setup_manifest.json').write_text(json.dumps(setup, indent=2))
    shutil.copy2(LOCAL_OUTPUT_ROOT / 'setup_manifest.json', DRIVE_OUTPUT_ROOT / 'setup_manifest.json')
    command = [PYTHON, HELPER, '--action', 'gpu', *COMMON, '--pairs', *GPU_PAIRS,
               '--max-batch', str(MAX_BATCH), '--memory-fraction', str(MEMORY_FRACTION)]
    if RESUME_GPU:
        command.append('--resume')
    run_command(command)
else:
    print('GPU/peak VRAM: NOT RUN — bật RUN_GPU_SMOKE khi muốn chạy trên Colab GPU.')

## 4. Tổng hợp và xuất Drive — không xếp hạng

In [ ]:
# @title Bảng 9 trạng thái kỹ thuật và manifest
import pandas as pd
run_command([PYTHON, HELPER, '--action', 'report', *COMMON])
table = pd.read_csv(LOCAL_OUTPUT_ROOT / 'adapter_preflight.csv', keep_default_na=False)
display(table)
summary = json.loads((LOCAL_OUTPUT_ROOT / 'adapter_preflight_summary.json').read_text())
print(json.dumps(summary, indent=2, ensure_ascii=False))
print('Kết quả bền vững:', DRIVE_OUTPUT_ROOT)
print('Manifest/config/commit/seed, logs và evidence từng cặp: run_manifest.json, cpu/, gpu/.')
print('params = Adapter; decoder_params và total_trainable_params được đếm riêng từ model thật.')
print('peak_vram_if_measured = peak RESERVED bytes; peak ALLOCATED và OOM/headroom trials nằm trong gpu/*.json.')
print('CPU PASS không chứng minh GPU PASS. r/d/Fusion vẫn pending joint selection D10.')
assert len(table) == 9
assert (table['status'] == 'PASS').all(), 'Có cặp FAIL; xem error/evidence.'

## 5. Chạy lại / giới hạn nghiệm thu

- CPU: `python scripts/run_g2.py --stage adapter_preflight --device cpu`; alias `adapter_screen` cũng chỉ chạy CPU preflight. Artifact mặc định `outputs/G2/D6/adapter_preflight.csv`.
- GPU CLI: `python scripts/g2_adapter_preflight.py --action gpu --model-config <yaml> --output-root <evidence> --pairs 256:768 --resume` sau CPU preflight với cùng config/source. Dùng `--pairs all` để kiểm tra 9 cặp trên GPU.
- Notebook không cần MVTec AD2 hay tar.gz cho D6 kỹ thuật. Không đọc DEV/TEST, không đo AU-PRO và không tạo checkpoint khoa học. Resume GPU dựa trên evidence từng cặp; CPU rất ngắn và được kiểm tra lại.
- E2 debug riêng vẫn có thể chạy bằng `scripts/run_g2.py --stage E2 --smoke --debug-pair 128 512 ...`, dùng pipeline cũ. Main chính thức chờ joint selection D10/D11; lock/checkpoint D6 cũ giữ nguyên để audit.
- Upload notebook này lên Colab **sau khi code mới đã commit/push vào `g2/member1`**; đặt SHA vào `GIT_COMMIT`. Đầu ra Drive sửa ở cell **OUTPUT**, GPU budget sửa ở cell **Input**.
